# Il segnale +1 / 0 / -1 per titolo (train e test)

**Cosa chiede il professore** ([resources/instructions.pdf](../resources/instructions.pdf), "Project Overview & Core Task"):
1. il sistema deve dare, per ogni titolo e in ogni momento, un segnale **+1** (long), **0** (fuori dal mercato) o **-1** (short);
2. il rendimento di ogni periodo è il **rendimento del titolo moltiplicato per il segnale**;
3. le ipotesi vanno dichiarate.

**Cosa facciamo qui.** La strategia lavora per coppie e il backtest ([backtest.py](backtest.py)) salva trade. Questo notebook trasforma gli stessi trade in un segnale per ciascuno dei 30 titoli, minuto per minuto, e ricalcola il guadagno con la formula del professore. Non cambia nessuna regola, nessun trade e nessuna decisione dell'agente. Le versioni sono quelle confrontate sul test: **A** (nessun agente), **B** e **C** di `finbert_episodio`, sia sul train sia sul test.

**I numeri principali restano quelli dei trade** (Passaggi 6 e 7 e i loro notebook sul test). Il calcolo dal segnale è la stessa strategia scritta nel formato del professore, e serve anche a controllare che i due calcoli coincidano. Questo notebook legge solo file già salvati: non rifà il test.

## Le ipotesi

- **Quando si decide e quando si esegue.** Il segnale si decide con i prezzi fino alla chiusura della barra da 1 minuto *t* e si esegue all'apertura della barra *t+1*: nessun dato futuro.
- **Da quale segnale.** Si entra quando lo z-score dello spread della coppia supera 2 in valore assoluto (dopo i primi 15 minuti e prima degli ultimi 30), si esce quando torna a 0, con lo stop a 3,5, dopo 90 minuti o a fine giornata. Nella B l'agente può bloccare l'ingresso; nella C può anche entrare nel verso opposto, seguendo la notizia. Regole complete in [signals.py](signals.py) e [docs/Test.md](../docs/Test.md).
- **Da coppia a titolo.** In un trade "long A / short B" il titolo A ha **+1** e il titolo B ha **-1**; fuori dai trade entrambi hanno **0**. Se due trade della stessa coppia fossero aperti insieme, le posizioni si sommerebbero (scelta del gruppo); con `finbert_episodio` non succede mai (sezione 3).
- **Rendimento di un periodo.** Il segnale della barra *t* vale per il rendimento semplice del titolo dall'apertura della barra *t* all'apertura della barra *t+1*; nell'ultima barra della giornata, dall'apertura alla chiusura. Di notte tutti i segnali sono 0: nessuna posizione aperta.
- **Importi.** 10.000 dollari su ogni titolo con segnale diverso da 0, senza leva; capitale di 20.000 dollari per coppia, 300.000 in tutto. Il rendimento giornaliero è il guadagno in dollari del giorno diviso 300.000, senza reinvestimento (come nel backtest).
- **Vendita allo scoperto** sempre possibile e senza costi di prestito. **Costo 0**, lo scenario principale (indicazione del docente).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # cartella del progetto: rende importabile src

import numpy as np
import pandas as pd

from src.config import RESULTS, RESULTS_TEST, TEST_END, TEST_START, TRAIN_END, VARIANTI
from src.backtest import CAPITAL_PER_PAIR, NOTIONAL_LEG, load_trades
from src.signals import list_pairs, load_pair

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
prezzi = {p: load_pair(p) for p in list_pairs().pair}          # prezzi a 1 minuto delle 15 coppie, train e test
CAPITALE = len(prezzi) * CAPITAL_PER_PAIR                         # 300.000 dollari
tutti = pd.DatetimeIndex(next(iter(prezzi.values())).ts.normalize().unique())
GIORNI = {"train": tutti[tutti <= TRAIN_END], "test": tutti[(tutti >= TEST_START) & (tutti <= TEST_END)]}
TRADE = {   # i trade salvati dal backtest, per periodo e versione
    "train": {"A": RESULTS / "trades_A.csv", "B": VARIANTI / "finbert_episodio/trades_B.csv", "C": VARIANTI / "finbert_episodio/trades_C.csv"},
    "test": {v: RESULTS_TEST / f"trades_{v}.csv" for v in "ABC"},
}
trade = {(per, v): load_trades(path, allow_test=per == "test") for per, d in TRADE.items() for v, path in d.items()}
print({k: len(t) for k, t in trade.items()}, f"| capitale {CAPITALE:,} $ | giorni: train {len(GIORNI['train'])}, test {len(GIORNI['test'])}")

{('train', 'A'): 973, ('train', 'B'): 968, ('train', 'C'): 969, ('test', 'A'): 500, ('test', 'B'): 493, ('test', 'C'): 495} | capitale 300,000 $ | giorni: train 64, test 32


## 1. Dai trade al segnale

Per ogni coppia si parte da un vettore di zeri, una posizione per barra. Per ogni trade si aggiunge +1 (long sul primo titolo della coppia) o -1 (short) dalla barra d'ingresso alla barra prima di quella d'uscita; se il trade esce a fine giornata, anche l'ultima barra, perché l'uscita è alla chiusura. Il secondo titolo della coppia ha il segnale opposto.

In [2]:
def segnale(trades: pd.DataFrame, giorni: pd.DatetimeIndex) -> pd.DataFrame:
    """Segnale per titolo (colonne) e per barra da 1 minuto (righe) nei giorni indicati, dai trade di una versione."""
    colonne = {}
    for p, d in prezzi.items():
        pos = np.zeros(len(d.ts), dtype=int)
        for t in trades[trades.pair == p].itertuples():
            e, k = d.ts.get_loc(t.t_entry), d.ts.get_loc(t.t_exit)
            fine = k + 1 if t.exit_reason == "fine_giornata" else k     # uscita alla chiusura di k, oppure all'apertura di k
            pos[e:fine] += 1 if t.direction.startswith("long") else -1
        tieni = d.ts.normalize().isin(giorni)
        colonne[d.sym_A] = pd.Series(pos[tieni], index=d.ts[tieni])
        colonne[d.sym_B] = pd.Series(-pos[tieni], index=d.ts[tieni])
    return pd.DataFrame(colonne).fillna(0).astype("int8").sort_index().rename_axis("ts")

## 2. Dal segnale al rendimento

Il rendimento semplice di ogni titolo per barra, come nelle ipotesi: dall'apertura della barra all'apertura della successiva, e nell'ultima barra della giornata dall'apertura alla chiusura. Il guadagno in dollari di una barra è la somma, sui 30 titoli, di segnale x rendimento x 10.000 dollari.

In [3]:
def rendimenti(giorni: pd.DatetimeIndex) -> pd.DataFrame:
    """Rendimento semplice di ogni titolo (colonne) per barra da 1 minuto (righe) nei giorni indicati."""
    colonne = {}
    for d in prezzi.values():
        tieni = d.ts.normalize().isin(giorni)
        for sym, apertura, chiusura in ((d.sym_A, d.oA, d.cA), (d.sym_B, d.oB, d.cB)):
            prossima = np.r_[apertura[1:], np.nan]
            r = np.where(d.last_of_day, chiusura / apertura - 1, prossima / apertura - 1)
            colonne[sym] = pd.Series(r[tieni], index=d.ts[tieni])
    return pd.DataFrame(colonne).sort_index().rename_axis("ts")


def guadagno_giornaliero(s: pd.DataFrame, r: pd.DataFrame, giorni: pd.DatetimeIndex) -> pd.DataFrame:
    """Guadagno in dollari e rendimento sul capitale per giorno, con la formula del professore. I giorni senza posizioni valgono 0."""
    pnl = (s * r[s.columns]).fillna(0).sum(axis=1) * NOTIONAL_LEG     # una coppia senza prezzi in un giorno ha segnale 0
    pnl = pnl.groupby(pnl.index.normalize()).sum().reindex(giorni, fill_value=0.0)
    return pd.DataFrame({"pnl_usd": pnl, "ret": pnl / CAPITALE}).rename_axis("day")

## 3. Il segnale di A, B e C, su train e test

Per ogni periodo e versione: quanti trade, in quanti minuti x titolo il segnale è diverso da 0, quali valori prende, e in quante barre due trade della stessa coppia sono aperti insieme (lì il segnale potrebbe valere +2 o -2).

In [4]:
segnali = {k: segnale(t, GIORNI[k[0]]) for k, t in trade.items()}


def sovrapposti(t: pd.DataFrame) -> int:
    """Trade che iniziano prima che il trade precedente della stessa coppia sia chiuso."""
    t = t.sort_values(["pair", "t_entry"])
    return int((t.t_entry < t.groupby("pair").t_exit.shift()).sum())


pd.DataFrame([dict(periodo=per, versione=v, trade=len(trade[(per, v)]), minuti_titolo_con_posizione=int((s != 0).sum().sum()),
                   valori=" ".join(map(str, sorted(np.unique(s.to_numpy())))), trade_sovrapposti=sovrapposti(trade[(per, v)]))
              for (per, v), s in segnali.items()]).set_index(["periodo", "versione"])

trade  minuti_titolo_con_posizione  valori  trade_sovrapposti
periodo versione                                                               
train   A           973                       122784  -1 0 1                  0
        B           968                       121884  -1 0 1                  0
        C           969                       122064  -1 0 1                  0
test    A           500                        63170  -1 0 1                  0
        B           493                        62186  -1 0 1                  0
        C           495                        62474  -1 0 1                  0

Un esempio: il segnale nei primi minuti in cui la A del test apre posizioni (solo i titoli con un segnale diverso da 0). Alle 9:46 del 17 settembre la strategia compra CRM e vende WDAY, vende KO e compra PEP, vende BAC e compra C.

In [5]:
s = segnali[("test", "A")]
primo = s.index[(s != 0).any(axis=1)][0]
esempio = s.loc[primo - pd.Timedelta(minutes=2): primo + pd.Timedelta(minutes=5)]
esempio.loc[:, (esempio != 0).any()]

,MA,V,CRM,WDAY,KO,PEP,BAC,C
ts,,,,,,,,
2026-09-17 09:44:00,0,0,0,0,0,0,0,0
2026-09-17 09:45:00,0,0,0,0,0,0,0,0
2026-09-17 09:46:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:47:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:48:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:49:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:50:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:51:00,1,-1,1,-1,-1,1,-1,1


## 4. Il rendimento con la formula del professore, accanto ai trade

Guadagno totale a costo 0 calcolato in due modi: sommando i trade del backtest (`pnl_trade_usd`, i numeri principali del progetto) e con segnale x rendimento del titolo (`pnl_segnale_usd`). Poi la differenza di B e C dalla A con ciascuno dei due calcoli.

Perché i due numeri non sono identici: il backtest misura il guadagno di un trade con i logaritmi dei prezzi e tiene fissi gli importi decisi all'ingresso; la formula del professore usa rendimenti semplici e pesi costanti minuto per minuto.

In [6]:
giornalieri, righe = {}, []
for per in GIORNI:
    r = rendimenti(GIORNI[per])
    for v in "ABC":
        g = guadagno_giornaliero(segnali[(per, v)], r, GIORNI[per])
        giornalieri[(per, v)] = g
        righe.append(dict(periodo=per, versione=v, pnl_trade_usd=trade[(per, v)].gross_bps.sum() / 1e4 * NOTIONAL_LEG, pnl_segnale_usd=g.pnl_usd.sum()))
confronto = pd.DataFrame(righe).set_index(["periodo", "versione"])
confronto["differenza_usd"] = confronto.pnl_segnale_usd - confronto.pnl_trade_usd
for col in ("pnl_trade_usd", "pnl_segnale_usd"):
    confronto[col.replace("pnl_", "delta_dalla_A_")] = confronto[col] - confronto[col].xs("A", level="versione").reindex(confronto.index, level="periodo")
confronto

pnl_trade_usd  pnl_segnale_usd  differenza_usd  delta_dalla_A_trade_usd  delta_dalla_A_segnale_usd
periodo versione                                                                                                    
train   A              7,239.43         7,245.45            6.02                     0.00                       0.00
        B              6,954.50         6,957.10            2.59                  -284.92                    -288.35
        C              6,588.66         6,597.63            8.98                  -650.77                    -647.82
test    A              2,472.42         2,477.40            4.98                     0.00                       0.00
        B              2,283.56         2,289.28            5.73                  -188.86                    -188.12
        C              1,229.96         1,232.21            2.24                -1,242.46                  -1,245.19

## 5. Salvataggio

- il segnale (minuti x 30 titoli) accanto ai trade di ogni versione: train in `Results/segnale_A.parquet` e `Results/varianti_agente/finbert_episodio/segnale_B.parquet`, `segnale_C.parquet`; test in `Results/test/segnale_A.parquet`, `segnale_B.parquet`, `segnale_C.parquet`;
- il rendimento giornaliero dal segnale di A, B e C: `Results/rendimenti_segnale.csv` (train) e `Results/test/rendimenti_segnale.csv` (test);
- la tabella del confronto, train e test: `Results/confronto_segnale_trade.csv`.

In [7]:
DOVE = {("train", "A"): RESULTS, ("train", "B"): VARIANTI / "finbert_episodio", ("train", "C"): VARIANTI / "finbert_episodio"} | {("test", v): RESULTS_TEST for v in "ABC"}
for (per, v), s in segnali.items():
    path = DOVE[(per, v)] / f"segnale_{v}.parquet"
    s.to_parquet(path)
    print(path.relative_to(RESULTS.parent), s.shape)
for per, cartella in (("train", RESULTS), ("test", RESULTS_TEST)):
    path = cartella / "rendimenti_segnale.csv"
    pd.concat({v: giornalieri[(per, v)].ret for v in "ABC"}, axis=1).to_csv(path)
    print(path.relative_to(RESULTS.parent))
confronto.to_csv(RESULTS / "confronto_segnale_trade.csv")
print((RESULTS / "confronto_segnale_trade.csv").relative_to(RESULTS.parent))

Results/segnale_A.parquet (24960, 30)
Results/varianti_agente/finbert_episodio/segnale_B.parquet (24960, 30)
Results/varianti_agente/finbert_episodio/segnale_C.parquet (24960, 30)
Results/test/segnale_A.parquet (12480, 30)
Results/test/segnale_B.parquet (12480, 30)
Results/test/segnale_C.parquet (12480, 30)
Results/rendimenti_segnale.csv
Results/test/rendimenti_segnale.csv
Results/confronto_segnale_trade.csv


## 6. Cosa ci dicono i risultati

**Il segnale c'è e ha solo tre valori.** Per ognuno dei 30 titoli e per ogni minuto, sul train (64 giorni) e sul test (32 giorni), il segnale vale -1, 0 o +1. Nessun trade di `finbert_episodio` si sovrappone a un altro della stessa coppia, né nella A né nella B né nella C: la regola della somma non scatta mai.

**La formula del professore dà gli stessi risultati dei trade.** La differenza va da +2 a +9 dollari su tutto il periodo, meno dello 0,3% del guadagno di ogni versione (per esempio la A del test: 2.472 dollari dai trade, 2.477 dal segnale).

**Le conclusioni non cambiano.** Le differenze di B e C dalla A sono le stesse a meno di 3 dollari:

| | Dai trade | Dal segnale |
|---|---|---|
| Test, B meno A | -189 $ | -188 $ |
| Test, C meno A | -1.242 $ | -1.245 $ |
| Train, B meno A | -285 $ | -288 $ |
| Train, C meno A | -651 $ | -648 $ |

Quello che i Passaggi 6 e 7 dicono sulla strategia e sull'agente vale anche con il segnale e la formula del professore.

**A cosa servono i file salvati.** I rendimenti giornalieri dal segnale (`rendimenti_segnale.csv`) sono la base per la curva di capitale da 100 con il benchmark e per le metriche annualizzate.

**Limiti.**
- Le stesse ipotesi del backtest: vendita allo scoperto sempre possibile e gratuita, esecuzione all'apertura della barra dopo senza effetto sul prezzo.
- Solo costo 0: con i costi il calcolo dal segnale andrebbe ripetuto addebitando ogni cambio di segnale.
- I prezzi intraday non sono aggiustati per i dividendi (Passaggio 2): lo stesso limite vale per il segnale.